# 岭回归与Lasso回归对比实验

## 目标

理解正则化方法的作用，对比岭回归和Lasso回归的特点。

- 理解L1和L2正则化的区别
- 对比岭回归和Lasso的回归系数变化
- 观察正则化强度的影响
- 在高维数据和共线性场景下对比效果

## 1. 导入库和生成数据

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pandas as pd

np.random.seed(42)

print("导入库完成")

## 2. 场景1: 共线性数据

In [ ]:
# 生成具有共线性的数据
n_samples = 100
n_features = 10

# 基础特征
X_base = np.random.randn(n_samples, 5)

# 创建共线性特征（特征的线性组合）
X = np.column_stack([
    X_base[:, 0],
    X_base[:, 1],
    X_base[:, 2],
    X_base[:, 3],
    X_base[:, 4],
    X_base[:, 0] * 0.9 + np.random.randn(n_samples) * 0.1,  # 与X1高度相关
    X_base[:, 1] * 0.9 + np.random.randn(n_samples) * 0.1,  # 与X2高度相关
    X_base[:, 0] * 0.5 + X_base[:, 1] * 0.5 + np.random.randn(n_samples) * 0.1,
    X_base[:, 2] * 0.8 + np.random.randn(n_samples) * 0.1,
    X_base[:, 3] * 0.8 + np.random.randn(n_samples) * 0.1
])

# 真实系数（只有前5个特征真正重要）
true_coef = np.array([3.0, -2.0, 1.5, 0.5, -1.0, 0, 0, 0, 0, 0])
true_intercept = 2.0

# 生成目标变量
y = np.dot(X, true_coef) + true_intercept + np.random.randn(n_samples) * 0.3

# 计算相关系数矩阵
corr_matrix = np.corrcoef(X.T)
print("相关系数矩阵 (前5x5):")
print(np.round(corr_matrix[:5, :5], 2))

# 显示高相关性
high_corr = []
for i in range(n_features):
    for j in range(i+1, n_features):
        if abs(corr_matrix[i, j]) > 0.8:
            high_corr.append((i, j, corr_matrix[i, j]))

print(f"\n发现 {len(high_corr)} 对高相关特征:")
for i, j, corr in high_corr:
    print(f"  特征{i+1} 和 特征{j+1}: r={corr:.3f}")

## 3. 数据标准化

In [ ]:
# 数据标准化（正则化需要标准化）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")

## 4. 训练不同模型

In [ ]:
# 训练线性回归
lr = LinearRegression()
lr.fit(X_train, y_train)

# 训练岭回归
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)

# 训练Lasso回归
lasso = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)

print("模型训练完成")

## 5. 对比回归系数

In [ ]:
# 创建系数对比表格
coef_df = pd.DataFrame({
    '真实系数': true_coef,
    '线性回归': lr.coef_,
    '岭回归(α=1.0)': ridge.coef_,
    'Lasso(α=0.1)': lasso.coef_
}, index=[f'特征{i+1}' for i in range(n_features)])

print("回归系数对比:")
print(coef_df.round(3))

# 计算系数的绝对值
print("\n系数绝对值:")
print(coef_df.abs().round(3))

## 6. 可视化回归系数

In [ ]:
plt.figure(figsize=(15, 5))

# 线性回归
plt.subplot(1, 3, 1)
plt.bar(range(n_features), lr.coef_, alpha=0.7, color='blue')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征')
plt.ylabel('系数值')
plt.title('线性回归系数\n(未正则化)')
plt.xticks(range(n_features), [f'F{i+1}' for i in range(n_features)], rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# 岭回归
plt.subplot(1, 3, 2)
plt.bar(range(n_features), ridge.coef_, alpha=0.7, color='green')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征')
plt.ylabel('系数值')
plt.title('岭回归系数 (L2正则化)\n(系数收缩)')
plt.xticks(range(n_features), [f'F{i+1}' for i in range(n_features)], rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# Lasso回归
plt.subplot(1, 3, 3)
plt.bar(range(n_features), lasso.coef_, alpha=0.7, color='red')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征')
plt.ylabel('系数值')
plt.title('Lasso回归系数 (L1正则化)\n(稀疏解)')
plt.xticks(range(n_features), [f'F{i+1}' for i in range(n_features)], rotation=45)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. 模型性能对比

In [ ]:
# 预测
y_pred_lr = lr.predict(X_test)
y_pred_ridge = ridge.predict(X_test)
y_pred_lasso = lasso.predict(X_test)

# 计算指标
results = {
    '模型': ['线性回归', '岭回归', 'Lasso'],
    '训练集R²': [
        r2_score(y_train, lr.predict(X_train)),
        r2_score(y_train, ridge.predict(X_train)),
        r2_score(y_train, lasso.predict(X_train))
    ],
    '测试集R²': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso)
    ],
    '测试集MSE': [
        mean_squared_error(y_test, y_pred_lr),
        mean_squared_error(y_test, y_pred_ridge),
        mean_squared_error(y_test, y_pred_lasso)
    ]
}

results_df = pd.DataFrame(results)
print("模型性能对比:")
print(results_df.round(4))

# 交叉验证
print("\n5折交叉验证R²得分:")
cv_lr = cross_val_score(lr, X_scaled, y, cv=5, scoring='r2')
cv_ridge = cross_val_score(ridge, X_scaled, y, cv=5, scoring='r2')
cv_lasso = cross_val_score(lasso, X_scaled, y, cv=5, scoring='r2')

print(f"线性回归: {cv_lr.mean():.4f} (±{cv_lr.std():.4f})")
print(f"岭回归: {cv_ridge.mean():.4f} (±{cv_ridge.std():.4f})")
print(f"Lasso: {cv_lasso.mean():.4f} (±{cv_lasso.std():.4f})")

## 8. 正则化强度的影响

In [ ]:
# 测试不同的正则化强度
alphas_ridge = np.logspace(-3, 3, 50)
alphas_lasso = np.logspace(-4, 1, 50)

ridge_coefs = []
ridge_scores = []

lasso_coefs = []
lasso_scores = []

for alpha in alphas_ridge:
    ridge_temp = Ridge(alpha=alpha, random_state=42)
    ridge_temp.fit(X_train, y_train)
    ridge_coefs.append(ridge_temp.coef_.copy())
    ridge_scores.append(ridge_temp.score(X_test, y_test))

for alpha in alphas_lasso:
    lasso_temp = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    lasso_temp.fit(X_train, y_train)
    lasso_coefs.append(lasso_temp.coef_.copy())
    lasso_scores.append(lasso_temp.score(X_test, y_test))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

# 可视化系数路径
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 岭回归系数路径
for i in range(n_features):
    axes[0, 0].plot(alphas_ridge, ridge_coefs[:, i], label=f'F{i+1}', alpha=0.7)
axes[0, 0].set_xscale('log')
axes[0, 0].set_xlabel('Alpha (对数尺度)')
axes[0, 0].set_ylabel('系数值')
axes[0, 0].set_title('岭回归系数路径')
axes[0, 0].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')
axes[0, 0].grid(True, alpha=0.3)

# Lasso系数路径
for i in range(n_features):
    axes[0, 1].plot(alphas_lasso, lasso_coefs[:, i], label=f'F{i+1}', alpha=0.7)
axes[0, 1].set_xscale('log')
axes[0, 1].set_xlabel('Alpha (对数尺度)')
axes[0, 1].set_ylabel('系数值')
axes[0, 1].set_title('Lasso回归系数路径')
axes[0, 1].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')
axes[0, 1].grid(True, alpha=0.3)

# 岭回归R²变化
axes[1, 0].plot(alphas_ridge, ridge_scores, 'b-', linewidth=2)
axes[1, 0].set_xscale('log')
axes[1, 0].set_xlabel('Alpha (对数尺度)')
axes[1, 0].set_ylabel('测试集 R²')
axes[1, 0].set_title('岭回归性能随Alpha变化')
axes[1, 0].grid(True, alpha=0.3)
best_ridge_alpha = alphas_ridge[np.argmax(ridge_scores)]
axes[1, 0].axvline(best_ridge_alpha, color='red', linestyle='--', 
               label=f'最佳α={best_ridge_alpha:.4f}')
axes[1, 0].legend()

# Lasso R²变化
axes[1, 1].plot(alphas_lasso, lasso_scores, 'r-', linewidth=2)
axes[1, 1].set_xscale('log')
axes[1, 1].set_xlabel('Alpha (对数尺度)')
axes[1, 1].set_ylabel('测试集 R²')
axes[1, 1].set_title('Lasso回归性能随Alpha变化')
axes[1, 1].grid(True, alpha=0.3)
best_lasso_alpha = alphas_lasso[np.argmax(lasso_scores)]
axes[1, 1].axvline(best_lasso_alpha, color='blue', linestyle='--',
               label=f'最佳α={best_lasso_alpha:.4f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"岭回归最佳Alpha: {best_ridge_alpha:.4f}")
print(f"Lasso最佳Alpha: {best_lasso_alpha:.4f}")

## 9. Lasso的特征选择能力

In [ ]:
# 统计不同alpha下被Lasso置零的特征数量
zero_features = []
for i, alpha in enumerate(alphas_lasso):
    zero_count = np.sum(lasso_coefs[i] == 0)
    zero_features.append(zero_count)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(alphas_lasso, zero_features, 'r-', linewidth=2)
plt.fill_between(alphas_lasso, zero_features, alpha=0.3)
plt.xscale('log')
plt.xlabel('Alpha (对数尺度)')
plt.ylabel('被置零的特征数量')
plt.title('Lasso的特征选择能力')
plt.grid(True, alpha=0.3)

# 在特定alpha下的系数分布
plt.subplot(1, 2, 2)
selected_alpha = best_lasso_alpha
lasso_selected = Lasso(alpha=selected_alpha, random_state=42, max_iter=10000)
lasso_selected.fit(X_train, y_train)

colors = ['green' if c != 0 else 'red' for c in lasso_selected.coef_]
plt.bar(range(n_features), lasso_selected.coef_, color=colors, alpha=0.7)
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征')
plt.ylabel('系数值')
plt.title(f'Lasso系数 (α={selected_alpha:.4f})\n绿色=保留, 红色=置零')
plt.xticks(range(n_features), [f'F{i+1}' for i in range(n_features)], rotation=45)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"在α={selected_alpha:.4f}时:")
print(f"保留特征数量: {np.sum(lasso_selected.coef_ != 0)}")
print(f"置零特征数量: {np.sum(lasso_selected.coef_ == 0)}")
print(f"\n保留的特征索引: {np.where(lasso_selected.coef_ != 0)[0] + 1}")

## 10. 场景2: 高维小样本数据

In [ ]:
# 生成高维小样本数据 (p > n)
n_samples_high = 50
n_features_high = 100

X_high = np.random.randn(n_samples_high, n_features_high)

# 只有10个特征真正重要
true_coef_high = np.zeros(n_features_high)
important_indices = np.random.choice(n_features_high, 10, replace=False)
true_coef_high[important_indices] = np.random.randn(10) * 2

y_high = np.dot(X_high, true_coef_high) + 2.0 + np.random.randn(n_samples_high) * 0.5

# 标准化
scaler_high = StandardScaler()
X_high_scaled = scaler_high.fit_transform(X_high)

X_train_high, X_test_high, y_train_high, y_test_high = train_test_split(
    X_high_scaled, y_high, test_size=0.3, random_state=42
)

print(f"高维数据: n={n_samples_high}, p={n_features_high}")
print(f"真正重要的特征数量: 10")
print(f"训练集: {X_train_high.shape}, 测试集: {X_test_high.shape}")

## 11. 高维数据下的模型对比

In [ ]:
# 线性回归（在p>n时不可行）
try:
    lr_high = LinearRegression()
    lr_high.fit(X_train_high, y_train_high)
    lr_r2 = lr_high.score(X_test_high, y_test_high)
    lr_train_r2 = lr_high.score(X_train_high, y_train_high)
    lr_works = True
except Exception as e:
    print(f"线性回归在高维数据下失败: {str(e)[:50]}...")
    lr_works = False

# 岭回归
ridge_high = Ridge(alpha=1.0, random_state=42)
ridge_high.fit(X_train_high, y_train_high)
ridge_r2 = ridge_high.score(X_test_high, y_test_high)
ridge_train_r2 = ridge_high.score(X_train_high, y_train_high)

# Lasso
lasso_high = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso_high.fit(X_train_high, y_train_high)
lasso_r2 = lasso_high.score(X_test_high, y_test_high)
lasso_train_r2 = lasso_high.score(X_train_high, y_train_high)

# 对比结果
print("高维数据下的模型性能:")
print("-" * 40)
if lr_works:
    print(f"线性回归:  训练R²={lr_train_r2:.4f}, 测试R²={lr_r2:.4f}")
print(f"岭回归:    训练R²={ridge_train_r2:.4f}, 测试R²={ridge_r2:.4f}")
print(f"Lasso:      训练R²={lasso_train_r2:.4f}, 测试R²={lasso_r2:.4f}")

print(f"\nLasso选择的特征数量: {np.sum(lasso_high.coef_ != 0)} / {n_features_high}")
print(f"岭回归系数绝对值的平均值: {np.mean(np.abs(ridge_high.coef_)):.6f}")

## 12. 可视化高维数据的系数

In [ ]:
plt.figure(figsize=(15, 6))

# 岭回归系数
plt.subplot(2, 2, 1)
plt.stem(range(n_features_high), ridge_high.coef_, markerfmt=' ', basefmt=' ')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征索引')
plt.ylabel('系数值')
plt.title('岭回归系数 (全部非零)')
plt.grid(True, alpha=0.3, axis='y')

# 岭回归系数（放大重要特征）
plt.subplot(2, 2, 2)
plt.stem(important_indices, ridge_high.coef_[important_indices], 
        linefmt='g-', markerfmt='go', basefmt=' ')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征索引')
plt.ylabel('系数值')
plt.title('岭回归系数 (真实重要特征)')
plt.grid(True, alpha=0.3, axis='y')

# Lasso系数
plt.subplot(2, 2, 3)
plt.stem(range(n_features_high), lasso_high.coef_, markerfmt=' ', basefmt=' ')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征索引')
plt.ylabel('系数值')
plt.title('Lasso系数 (稀疏)')
plt.grid(True, alpha=0.3, axis='y')

# Lasso选中的特征 vs 真实重要特征
plt.subplot(2, 2, 4)
selected_by_lasso = np.where(np.abs(lasso_high.coef_) > 1e-6)[0]
plt.stem(selected_by_lasso, lasso_high.coef_[selected_by_lasso],
        linefmt='r-', markerfmt='ro', basefmt=' ', label='Lasso选择')
plt.stem(important_indices, true_coef_high[important_indices],
        linefmt='g--', markerfmt='gx', basefmt=' ', label='真实重要')
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('特征索引')
plt.ylabel('系数值')
plt.title('Lasso选择 vs 真实重要特征')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 计算召回率
true_set = set(important_indices)
selected_set = set(selected_by_lasso)
recall = len(true_set & selected_set) / len(true_set)
precision = len(true_set & selected_set) / len(selected_set) if len(selected_set) > 0 else 0

print(f"\nLasso特征选择质量:")
print(f"真实重要特征: {len(true_set)}个")
print(f"Lasso选择特征: {len(selected_set)}个")
print(f"正确识别: {len(true_set & selected_set)}个")
print(f"召回率: {recall:.2%}")
print(f"精确率: {precision:.2%}")

## 总结

### 岭回归 (L2正则化) vs Lasso回归 (L1正则化)

| 特性 | 岭回归 (L2) | Lasso回归 (L1) |
|------|-------------|---------------|
| 惩罚项 | Σβ² | Σ|β| |
| 系数变化 | 收缩至接近0 | 精确地置为0 |
| 特征选择 | 否 | 是 |
| 共线性处理 | 好 | 会随机选择一个 |
| 适用场景 | 所有特征都重要 | 只有小部分特征重要 |
| 解的唯一性 | 是 | 可能不唯一 |

### 关键要点

1. **岭回归 (L2)**:
   - 系数均匀收缩，不会变为零
   - 适合处理共线性问题
   - 所有特征都保留，适合解释性要求不高的场景

2. **Lasso回归 (L1)**:
   - 产生稀疏解，自动进行特征选择
   - 适合高维小样本数据 (p > n)
   - 适合只有少数特征真正重要的场景

3. **正则化强度 (α)**:
   - α越大，正则化越强
   - 需要通过交叉验证选择最佳α
   - 过大会导致欠拟合，过小会导致过拟合

4. **数据标准化**:
   - 正则化前必须标准化
   - 不同尺度的特征会受不同惩罚

### 实践建议

- 默认使用Ridge，除非需要特征选择
- 如果特征数远大于样本数，考虑Lasso
- 如果不确定，可以尝试Elastic Net (L1+L2组合)
- 始终使用交叉验证选择正则化强度